# 6주차 데이터 시퀀스 생성

In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = '../../tests/data_/returns.csv'
returns = pd.read_csv(file_path, index_col=0, parse_dates=True)

print(returns.shape)

(1826, 40)


In [3]:
sp500_tickers = [
    'NVDA',   # 엔비디아
    'AAPL',   # 애플
    'MSFT',   # 마이크로소프트
    'AMZN',   # 아마존
    'GOOGL',  # 알파벳 A
    'META',   # 메타 플랫폼스
    'BRK-B',  # 버크셔 해서웨이
    'LLY',    # 일라이 릴리
    'AVGO',   # 브로드컴
    'TSLA',   # 테슬라
    'WMT',    # 월마트
    'JPM',    # JP모건 체이스
    'UNH',    # 유나이티드헬스 그룹
    'XOM',    # 엑슨모빌
    'V',      # 비자
    'MA',     # 마스터카드
    'JNJ',    # 존슨앤드존슨
    'PG',     # 프록터 앤 갬블
    'ORCL',   # 오라클
    'HD',     # 홈디포
    'COST',   # 코스트코
    'ABBV',   # 애비비
    'BAC',    # 뱅크오브아메리카
    'NFLX',   # 넷플릭스
    'CVX',    # 쉐브론
    'AMD',    # AMD
    'KO',     # 코카콜라
    'CAT',    # 캐터필러
    'PLTR',   # 팔란티어
    'MU'      # 마이크론 테크놀로지
]

crypto_tickers = [
    'BTC-USD',   # 1. 비트코인 
    'ETH-USD',   # 2. 이더리움 
    'USDT-USD',  # 3. 테더 
    'BNB-USD',   # 4. 바이낸스 코인
    'XRP-USD',   # 5. 리플 
    'USDC-USD',  # 6. 유에스디 코인
    'SOL-USD',   # 7. 솔라나 
    'TRX-USD',   # 8. 트론
    'DOGE-USD',  # 9. 도지코인 
    'ADA-USD'    # 10. 카르다노
]


In [4]:
crypto_returns = returns[crypto_tickers]
stock_returns = returns[sp500_tickers]

crypto_medians = crypto_returns.median(axis=1)
stock_medians = stock_returns.median(axis=1)

targets_crypto = crypto_returns.apply(lambda x: (x >= crypto_medians).astype(int))
targets_stock = stock_returns.apply(lambda x: (x >= stock_medians).astype(int))

# 라벨링 끝난 정답지 다시 하나로 합치기 (나중에 모델에 넣기 위함)
targets = pd.concat([targets_stock, targets_crypto], axis=1)

In [ ]:
import pandas as pd
import numpy as np

def create_dataframe_rolling_sequences(crypto_ret, stock_ret, targets_df, seq_length, window_size=608, test_size=152, step_size=152):
    all_rows = [] # 최종적으로 데이터프레임을 만들기 전, 각각의 완성된 행을 담을 리스트
    total_length = len(crypto_ret)
    train_size = window_size - test_size # 456일
    
    start_idx = 0
    split_count = 1
    
    # 돋보기(window_size)가 전체 데이터 끝을 넘어가지 않을 때까지 반복
    while start_idx + window_size <= total_length: #윈도우 사이즈 608일을 잡았을때 전체 테이터 길이를 안넘는다면
        # 1. 구간 인덱스 설정
        train_start = start_idx
        train_end = train_start + train_size
        test_end = train_start + window_size
        
        # 2. Train 구간 자르기
        train_crypto = crypto_ret.iloc[train_start:train_end]
        train_stock = stock_ret.iloc[train_start:train_end]
        
        # 3. [미래 참조 방지] Train 데이터로만 평균/표준편차 구하기
        c_mean, c_std = train_crypto.mean(), train_crypto.std()
        s_mean, s_std = train_stock.mean(), train_stock.std()
        
        # 4. 정규화
        norm_crypto = (crypto_ret.iloc[train_start:test_end] - c_mean) / c_std
        norm_stock = (stock_ret.iloc[train_start:test_end] - s_mean) / s_std
        
        norm_all = pd.concat([norm_stock, norm_crypto], axis=1)
        target_chunk = targets_df.iloc[train_start:test_end]
        
        # 날짜 인덱스 추출 (데이터프레임에 타겟 날짜를 기록하기 위함)
        dates = norm_all.index
        
        # 5. 슬라이딩 윈도우 생성 및 행(Row) 데이터 추출
        for ticker in norm_all.columns:
            asset_X = norm_all[ticker].values #입력할 수익률 데이터
            asset_y = target_chunk[ticker].values # 맞춰야할 정답 레이블
            
            for j in range(len(asset_X) - seq_length):
                window = asset_X[j : j + seq_length]
                target = asset_y[j + seq_length]
                target_date = dates[j + seq_length]
                
                # 예측 대상일이 456일 경계선 이전이면 Train, 이후면 Test
                dataset_type = 'Train' if (j + seq_length) < train_size else 'Test'
                
                # 기본 정보 딕셔너리 생성
                row_data = {
                    'split_id': split_count,
                    'dataset_type': dataset_type,
                    'ticker': ticker,
                    'target_date': target_date,
                }
                
                # 시퀀스 데이터를 t-N 컬럼으로 펼쳐서 추가 (t-30 ~ t-1)
                seq_dict = {f't-{seq_length - k}': window[k] for k in range(seq_length)}
                row_data.update(seq_dict)
                
                # 최종 정답(Target) 추가
                row_data['target'] = target
                
                # 리스트에 적재
                all_rows.append(row_data)
                
        print(f"✅ Split {split_count} 처리 완료: {train_start}일 ~ {test_end}일")
        
        start_idx += step_size
        split_count += 1
        
    print("🔥 단일 데이터프레임으로 병합 중...")
    # 6. 모인 데이터를 단일 DataFrame으로 변환
    final_df = pd.DataFrame(all_rows)
    
    return final_df

# 함수 실행 및 결과 저장
print("롤링 데이터프레임 생성 시작")
final_sequence_df = create_dataframe_rolling_sequences(crypto_returns, stock_returns, targets, 30, window_size=608, test_size=152, step_size=152)

# 결과 확인
print(f"\n최종 생성된 데이터프레임 형태: {final_sequence_df.shape}")
display(final_sequence_df.head())

롤링 데이터프레임 생성 시작
✅ Split 1 처리 완료: 0일 ~ 608일
✅ Split 2 처리 완료: 152일 ~ 760일
✅ Split 3 처리 완료: 304일 ~ 912일
✅ Split 4 처리 완료: 456일 ~ 1064일
✅ Split 5 처리 완료: 608일 ~ 1216일
✅ Split 6 처리 완료: 760일 ~ 1368일
✅ Split 7 처리 완료: 912일 ~ 1520일
✅ Split 8 처리 완료: 1064일 ~ 1672일
✅ Split 9 처리 완료: 1216일 ~ 1824일
🔥 단일 데이터프레임으로 병합 중...

최종 생성된 데이터프레임 형태: (208080, 35)


,split_id,dataset_type,ticker,target_date,t-30,t-29,t-28,t-27,t-26,t-25,...,t-9,t-8,t-7,t-6,t-5,t-4,t-3,t-2,t-1,target
0,1,Train,NVDA,2021-05-09,0.179889,-0.022025,-0.022025,1.934757,1.055462,-0.916125,...,-0.738527,-0.022025,-0.022025,-0.422895,-1.161767,0.238271,0.133356,0.671674,-0.022025,1
1,1,Train,NVDA,2021-05-10,-0.022025,-0.022025,1.934757,1.055462,-0.916125,1.939273,...,-0.022025,-0.022025,-0.422895,-1.161767,0.238271,0.133356,0.671674,-0.022025,-0.022025,0
2,1,Train,NVDA,2021-05-11,-0.022025,1.934757,1.055462,-0.916125,1.939273,-0.507128,...,-0.022025,-0.422895,-1.161767,0.238271,0.133356,0.671674,-0.022025,-0.022025,-1.307094,1
3,1,Train,NVDA,2021-05-12,1.934757,1.055462,-0.916125,1.939273,-0.507128,-0.022025,...,-0.422895,-1.161767,0.238271,0.133356,0.671674,-0.022025,-0.022025,-1.307094,0.076858,0
4,1,Train,NVDA,2021-05-13,1.055462,-0.916125,1.939273,-0.507128,-0.022025,-0.022025,...,-1.161767,0.238271,0.133356,0.671674,-0.022025,-0.022025,-1.307094,0.076858,-1.355584,0


In [6]:
import matplotlib.pyplot as plt
import random
import numpy as np

# 1. 연구 세트 선택 (0번 ~ 8번 입력 가능)
target_split = 0  

# 변수명은 현재 가지고 계신 데이터 변수명
X_train_split, y_train_split, _, _ = splits_30_paper[target_split]

# 2. 랜덤으로 문제 번호 하나 뽑기
random_idx = random.randint(0, len(X_train_split) - 1)

# 3. 데이터 가져오기
sample_X = X_train_split[random_idx].flatten() 
sample_y = y_train_split[random_idx]

# 데이터의 실제 길이를 자동으로 파악 (30이면 30, 90이면 90)
seq_len = len(sample_X)

# 4. 정답 라벨 변환
target_text = "BULL (1)" if sample_y == 1 else "BEAR (0)"

# 5. 그래프 그리기
plt.figure(figsize=(10, 5))

# X축과 데이터를 맞춤!
plt.plot(range(1, seq_len + 1), sample_X, marker='o', linestyle='-', color='b', markersize=4, label=f'Past {seq_len} Days (X)')

plt.axhline(0, color='red', linestyle='--', alpha=0.7, label='Train Mean (0)')

# 타이틀과 축도 시퀀스 길이에 맞춰서 자동 출력
plt.title(f"[Split #{target_split + 1} | Sample #{random_idx}] Target for Day {seq_len + 1}: {target_text}", fontsize=14, fontweight='bold')
plt.xlabel("Time Step (Days)", fontsize=12)
plt.ylabel("Normalized Return", fontsize=12)

# X축 눈금이 90개면 너무 지저분하니까 10일 단위 정도로 큼직하게 표시 (원리: seq_len을 10등분)
step = max(1, seq_len // 10) 
plt.xticks(range(1, seq_len + 1, step)) 

plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

NameError: name 'splits_30_paper' is not defined